# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² colorectal cancer dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant pandas

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\n\n{metadata.description}")

## 2. Data Overview
List available record sets, their `@id`, and fields.

**Note:** All references to entities use their `@id` for clarity and consistency.

In [ ]:
# List all available record sets and their fields by @id
print('Available Record Sets:')
record_set_objs = list(dataset.record_sets)
record_set_ids = []
for rs in record_set_objs:
    print(f"  - Record set @id: {rs.id} | Name: {rs.name}")
    record_set_ids.append(rs.id)
    print("    Fields:")
    for field in rs.fields:
        print(f"      * Field @id: {field.id} | Name: {field.name} | Data type: {field.data_type}")
    print()

## 3. Data Extraction
Load data from each available record set into pandas DataFrames. Use record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set using their @id
dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nExtracting data for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns: {list(df.columns)}")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Let's perform some basic EDA: filtering, normalization, and grouping by key fields.

Use field and record set `@id`s as referenced earlier.

In [ ]:
# For demonstration, choose the first available record set containing numeric fields
selected_record_set = None
numeric_field = None
group_field = None
for rs_id, df in dataframes.items():
    if not df.empty:
        # Find numeric fields
        for col in df.columns:
            # Try to detect float/int columns
            sample = pd.to_numeric(df[col], errors='coerce')
            # If at least 80% non-null --> treat as candidate numeric
            if sample.notnull().sum() / len(df[col]) > 0.8:
                numeric_field = col
                selected_record_set = rs_id
                break
        # Find a group field as a categorical (non-numeric, low cardinality)
        for col in df.columns:
            if col != numeric_field:
                unique_vals = df[col].nunique(dropna=True)
                if unique_vals < 10 and unique_vals > 1:
                    group_field = col
                    break
        if numeric_field:
            break
if selected_record_set is None:
    print("No suitable numeric field found for EDA.")
else:
    print(f"Selected record set @id: {selected_record_set}")
    print(f"Numeric field @id: {numeric_field}")
    if group_field:
        print(f"Group field @id: {group_field}")
    df = dataframes[selected_record_set].copy()
    # Ensure numeric
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

    # Remove non-finite
    df = df[df[numeric_field].notnull() & np.isfinite(df[numeric_field])]
    # Define a threshold as the 75th percentile (for demonstration)
    threshold = df[numeric_field].quantile(0.75)
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"\nFiltered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by group_field if available
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nMean {numeric_field} grouped by {group_field}:")
        display(grouped_df)

## 5. Visualization
Visualize the distribution of the chosen numeric field and its relationship with a grouping field, if available.

In [ ]:
# Histogram of the numeric field in the selected record set
if selected_record_set and numeric_field:
    df = dataframes[selected_record_set]
    values = pd.to_numeric(df[numeric_field], errors='coerce')
    plt.figure(figsize=(7, 4))
    plt.hist(values.dropna(), bins=15, color='steelblue', alpha=0.8)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    
    if group_field:
        plt.figure(figsize=(7, 4))
        df_box = df[[numeric_field, group_field]].copy()
        df_box[numeric_field] = pd.to_numeric(df_box[numeric_field], errors='coerce')
        df_box = df_box.dropna(subset=[numeric_field, group_field])
        df_box.boxplot(column=numeric_field, by=group_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.suptitle('')
        plt.show()

## 6. Conclusion
This notebook demonstrated how to use the `mlcroissant` library to load, explore, and analyze the FAIR² colorectal cancer survivors dataset. By referencing record sets and fields via their `@id`, you can reliably access and process data for transparent, reproducible science.

**Key steps:**
- Loaded Croissant metadata and discovered available record sets/fields (`@id`-based).
- Extracted tabular data for analysis using pandas.
- Performed basic EDA: numeric field filtering, normalization, and simple grouping.
- Visualized data distributions.

For deeper analyses, explore combining fields/record sets, or integrating other FAIR datasets via their Croissant schema!